# Executive Dashboard
This notebook reads the output tables from NB_07 (convergence query)
and presents findings in a format suitable for a non-technical audience:
plan sponsors, account managers, compliance officers, and executive
leadership.

The individual audit notebooks (NB_01 through NB_05) answer technical
questions about what went wrong. This notebook answers operational
questions about what to do about it:

| Section | Question it answers |
|---------|-------------------|
| Pipeline health | Did the pipeline run cleanly? Any notebooks fail? |
| KPI scorecard | How many findings, how much exposure, how many members? |
| Severity breakdown | Where should we focus first? |
| Category analysis | Which audit domains are producing the most findings? |
| Plan-level exposure | Which employer groups need remediation conversations? |
| Multi-category members | Which members have the most complex, overlapping issues? |
| Actionable next steps | What should operations do on Monday morning? |

The dashboard uses Databricks `display()` for native rendering so the
charts work directly in the notebook output and in scheduled email reports.

In [0]:
%run ./NB_00_pipeline_utilities

# NB00 - Pipeline Utilities
**Eligibility & Accumulator**

In production, a reconciliation pipeline has dozens of checks spread across multiple notebooks.
If one those check fails, say the TPA eligibility table got dropped by an upstream job, you don't
want the entire pipeline to crash and leave you with nothing. You still want the accumulator
reconciliation, the copay audit, and the rollover checks to run and produce findings.

You also need to know exactly what ran, what passed, and what eventually broke.

This notebook provides shared utilities that every downstream notebook imports via
`%run ./NB_00_pipeline_utilities`. It handles three things:

| Utility | What it does |
|---------|-------------|
| Structured logging | JSON audit entries to `pbm_sim.pipeline_audit_log`, queryable by run, notebook, section |
| Error handling decorator | Wraps each check in try/except so one failure doesn't kill the entire pipeline |
| Table validation helpers | Pre-flight checks before a notebook starts its logic (do the table exists? does it has rows? does it have right columns?) |

These functions are used in NB01 through NB07.

## Structured Logging

Every pipeline run gets a unique `run_id` based on the UTC timestamp when the run started.
It ties together all the log entries from a single execution across notebooks.
If the orchestrator kicks off NB01 through NB07 sequentially, they all share the same run_id.

Log entries go to two places:
1. **stdout** as `[PIPELINE_LOG] {...}` - captured by Databricks job logs, useful for real-time monitoring
2. **Delta table** `pbm_sim.pipeline_audit_log` - the permanent audit trail you can query later

Each entry records the notebook, section, outcome (SUCCESS/WARNING/ERROR/SKIPPED),
row count, timing, and a free-text detail field. The detail field is truncated at 2,000
characters to prevent a full stack trace from bloating the table.

## Error Handling Decorator

This is the core safety mechanism. In a reconciliation pipeline, each notebook runs
multiple independent checks. Take NB03 (accumulator reconciliation) - it checks for
deductible overcharges AND OOPM overcharges. If the deductible check throws an exception
(maybe a column was renamed upstream), you still want the OOPM check to run and produce
its findings. Without this decorator, an unhandled exception in the deductible check
would kill the notebook, and you'd lose the OOPM results too.

The decorator does three things on failure:
1. Catches the exception and logs it as ERROR with the full traceback
2. Returns an empty DataFrame with the expected schema so downstream unions don't break
3. Lets the notebook continue to the next section

On success, it logs the section as SUCCESS with the row count and wall-clock time.

**Usage in a downstream notebook:**
```python
@pipeline_section("NB_03", "deductible_overcharge", finding_schema=convergence_schema)
def run_deductible_check():
    # ... compute df ...
    return df

df_result = run_deductible_check()
# If the check fails, df_result is an empty DataFrame with the right columns
# so the union at the end of the notebook still works
```

## Table Validation Helpers (Pre-flight Checks)

Before a notebook starts its actual logic, it should verify that the tables it depends on
exist and have data. Without this, a missing table produces a cryptic *AnalysisException*
50 cells into the notebook, and you spend 10 minutes figuring out that the problem was
simply that NB00 didn't run.

These are the pre-flight checks:
- **The Table exists?** Catches the case where the upstream notebook didn't run or the table got dropped
- **Are the table empty?** The table exists but has zero rows - maybe the data generator ran but the filter
  produced nothing. This is different from "table doesn't exist" and needs a different response
- **Are the required columns present?** Schema drift - a column was renamed or removed upstream. Better to catch
  this up front with a clear message than to get a "Column 'xyz' not found" error mid-analysis

## Pipeline Metadata

Convenience function for embedding run metadata into output tables.
When NB07 unions all findings, each row can carry the run_id so you can
trace any finding back to the exact pipeline execution that produced it.

Pipeline utilities loaded. Run ID: RUN_20261019_160540


In [0]:
from pyspark.sql import functions as F

spark.sql("USE pbm_sim")

NB = "NB_08"

# Pre-flight: we need the three output tables from NB_07.
# If NB_07 didn't run or failed, there's nothing to display.
for tbl in ["pbm_sim.findings_registry", "pbm_sim.findings_member_rollup", "pbm_sim.findings_plan_rollup"]:
    if not validate_table_exists(tbl, NB):
        raise RuntimeError(f"Required table {tbl} not found - NB_07 must run before NB_08")

df_registry = spark.table("pbm_sim.findings_registry")
df_member   = spark.table("pbm_sim.findings_member_rollup")
df_plan     = spark.table("pbm_sim.findings_plan_rollup")

log_entry(NB, "load_dashboard_tables", "SUCCESS",
          detail=f"registry={df_registry.count()}, members={df_member.count()}, plans={df_plan.count()}")
print("Dashboard source tables loaded.")

[PIPELINE_LOG] {"run_id": "RUN_20261019_160540", "timestamp": "2026-10-19T16:06:13.195812+00:00", "notebook": "NB_08", "section": "load_dashboard_tables", "status": "SUCCESS", "row_count": 0, "detail": "registry=7084, members=5850, plans=40", "duration_sec": 0.0}
Dashboard source tables loaded.


## Pipeline Health

Before looking at findings, confirm the pipeline ran cleanly.
A failed notebook means the convergence data is partial, and the
numbers below may undercount exposure in that domain.

In [0]:
# Pull the most recent pipeline run from the audit log.
# This tells us which notebooks succeeded and which failed,
# so the reader knows if any findings category is missing.
try:
    df_log = spark.table("pbm_sim.pipeline_audit_log")

    # Get the latest run_id
    latest_run = (
        df_log
        .select("run_id")
        .orderBy(F.col("timestamp").desc())
        .first()
    )

    if latest_run:
        run_id = latest_run["run_id"]
        df_run_log = (
            df_log
            .filter(F.col("run_id") == run_id)
            .orderBy("timestamp")
        )

        # Count by status
        status_counts = {
            row["status"]: row["count"]
            for row in df_run_log.groupBy("status").count().collect()
        }

        print(f"Latest pipeline run: {run_id}")
        print(f"  Sections logged: {df_run_log.count()}")
        print(f"  SUCCESS: {status_counts.get('SUCCESS', 0)}")
        print(f"  WARNING: {status_counts.get('WARNING', 0)}")
        print(f"  ERROR:   {status_counts.get('ERROR', 0)}")

        errors = df_run_log.filter(F.col("status") == "ERROR")
        if errors.count() > 0:
            print()
            print("  Failed sections:")
            for row in errors.collect():
                print(f"    [{row['notebook']}] {row['section']}")
        else:
            print("  All sections completed successfully.")

except Exception:
    print("  (Audit log not available - pipeline health unknown)")

log_entry(NB, "pipeline_health", "SUCCESS")

Latest pipeline run: RUN_20261019_160540
  Sections logged: 1
  SUCCESS: 1
  ERROR:   0
  All sections completed successfully.
[PIPELINE_LOG] {"run_id": "RUN_20261019_160540", "timestamp": "2026-10-19T16:06:37.932140+00:00", "notebook": "NB_08", "section": "pipeline_health", "status": "SUCCESS", "row_count": 0, "detail": "", "duration_sec": 0.0}


## KPI Scorecard

The four numbers that matter most in a reconciliation pipeline review.
These go at the top of any status report or executive briefing.

In [0]:
total_findings = df_registry.count()
total_members = df_registry.filter(F.col("member_id").isNotNull()).select("member_id").distinct().count()
total_exposure = df_registry.agg(F.sum("dollar_exposure")).collect()[0][0] or 0
critical_count = df_registry.filter(F.col("severity") == "CRITICAL").count()
high_count = df_registry.filter(F.col("severity") == "HIGH").count()

print("=" * 65)
print("  RECONCILIATION KPI SCORECARD")
print("=" * 65)
print(f"  Total findings:                {total_findings:,}")
print(f"  Unique members affected:       {total_members:,}")
print(f"  Total dollar exposure:         ${total_exposure:,.2f}")
print(f"  CRITICAL findings:             {critical_count:,}")
print(f"  HIGH findings:                 {high_count:,}")
print(f"  Combined CRITICAL + HIGH:      {critical_count + high_count:,} ({(critical_count + high_count) / max(total_findings, 1):.1%} of all)")
print("=" * 65)

log_entry(NB, "kpi_scorecard", "SUCCESS",
          detail=f"findings={total_findings}, members={total_members}, exposure=${total_exposure:,.2f}")

  RECONCILIATION KPI SCORECARD
  Total findings:                7,084
  Unique members affected:       5,850
  Total dollar exposure:         $3,688,214.20
  CRITICAL findings:             2,556
  HIGH findings:                 4,277
  Combined CRITICAL + HIGH:      6,833 (96.5% of all)
[PIPELINE_LOG] {"run_id": "RUN_20261019_160540", "timestamp": "2026-10-19T16:06:47.700685+00:00", "notebook": "NB_08", "section": "kpi_scorecard", "status": "SUCCESS", "row_count": 0, "detail": "findings=7084, members=5850, exposure=$3,688,214.20", "duration_sec": 0.0}


## Severity Distribution

How findings break down by severity level. CRITICAL findings need
immediate attention (active overcharges, compliance violations).
HIGH findings need investigation within the week. MEDIUM findings
are data quality issues to address in the normal review cycle.

In [0]:
df_severity = (
    df_registry
    .groupBy("severity")
    .agg(
        F.count("*").alias("findings"),
        F.countDistinct("member_id").alias("members_affected"),
        F.round(F.sum("dollar_exposure"), 2).alias("dollar_exposure"),
    )
    .withColumn(
        "sort_order",
        F.when(F.col("severity") == "CRITICAL", 1)
        .when(F.col("severity") == "HIGH", 2)
        .when(F.col("severity") == "MEDIUM", 3)
        .otherwise(4),
    )
    .orderBy("sort_order")
    .drop("sort_order")
)

print("Findings by severity:")
display(df_severity)

log_entry(NB, "severity_distribution", "SUCCESS")

Findings by severity:


severity,findings,members_affected,dollar_exposure
CRITICAL,2556,2556,714720.89
HIGH,4277,3207,2973131.02
MEDIUM,251,250,362.29


[PIPELINE_LOG] {"run_id": "RUN_20261019_160540", "timestamp": "2026-10-19T16:06:51.820268+00:00", "notebook": "NB_08", "section": "severity_distribution", "status": "SUCCESS", "row_count": 0, "detail": "", "duration_sec": 0.0}


## Findings by Category

Which audit domains are producing the most findings? A category with
a high count and high dollar exposure is where the team should focus
remediation efforts first. A category with a high count but low dollar
exposure (like FILE_QUALITY) may indicate a systemic upstream data
problem that needs a fix at the source rather than per-finding remediation.

In [0]:
df_by_category = (
    df_registry
    .groupBy("issue_category")
    .agg(
        F.count("*").alias("findings"),
        F.countDistinct("member_id").alias("members_affected"),
        F.countDistinct("issue_type").alias("issue_types"),
        F.round(F.sum("dollar_exposure"), 2).alias("dollar_exposure"),
        F.sum(F.when(F.col("severity") == "CRITICAL", 1).otherwise(0)).alias("critical"),
        F.sum(F.when(F.col("severity") == "HIGH", 1).otherwise(0)).alias("high"),
    )
    .orderBy(F.col("dollar_exposure").desc())
)

print("Findings by audit category:")
display(df_by_category)

Findings by audit category:


issue_category,findings,members_affected,issue_types,dollar_exposure,critical,high
COPAY_PROGRAM,3756,2719,2,2973131.02,0,3756
ACCUMULATOR,2056,2056,2,714720.89,2056,0
ROLLOVER,415,415,2,362.29,0,375
ELIGIBILITY,825,822,3,0.0,500,125
FILE_QUALITY,32,10,4,0.0,0,21


## Issue Type Detail

Drill-down from category to specific issue type. This is the level
of detail that the reconciliation team needs to prioritize their
work queue.

In [0]:
df_by_type = (
    df_registry
    .groupBy("issue_category", "issue_type", "severity")
    .agg(
        F.count("*").alias("findings"),
        F.countDistinct("member_id").alias("members"),
        F.round(F.sum("dollar_exposure"), 2).alias("dollar_exposure"),
    )
    .withColumn(
        "sev_order",
        F.when(F.col("severity") == "CRITICAL", 1)
        .when(F.col("severity") == "HIGH", 2)
        .when(F.col("severity") == "MEDIUM", 3)
        .otherwise(4),
    )
    .orderBy("sev_order", F.col("dollar_exposure").desc())
    .drop("sev_order")
)

print("Findings by issue type:")
display(df_by_type)

log_entry(NB, "category_analysis", "SUCCESS")

Findings by issue type:


issue_category,issue_type,severity,findings,members,dollar_exposure
ACCUMULATOR,DEDUCTIBLE_OVERCHARGE,CRITICAL,1681,1681,554713.64
ACCUMULATOR,OOPM_OVERCHARGE,CRITICAL,375,375,160007.25
ELIGIBILITY,TERM_MISMATCH,CRITICAL,500,500,0.0
COPAY_PROGRAM,MAXIMIZER_PRE_EXHAUSTION_ERROR,HIGH,3696,2659,2924556.94
COPAY_PROGRAM,COUPON_INCORRECTLY_CREDITED,HIGH,60,60,48574.08
FILE_QUALITY,STALE_FILE,HIGH,1,0,0.0
ROLLOVER,ROLLOVER_FAILURE,HIGH,375,375,0.0
FILE_QUALITY,NULL_MEMBER_ID,HIGH,20,0,0.0
ELIGIBILITY,MISSING_FROM_TPA,HIGH,125,125,0.0
ROLLOVER,CROSS_YEAR_MISATTRIBUTION,MEDIUM,40,40,362.29


[PIPELINE_LOG] {"run_id": "RUN_20261019_160540", "timestamp": "2026-10-19T16:06:59.869475+00:00", "notebook": "NB_08", "section": "category_analysis", "status": "SUCCESS", "row_count": 0, "detail": "", "duration_sec": 0.0}


## Top Plans by Dollar Exposure

Which employer groups have the highest financial exposure? These are
the plans where the account management team needs to start remediation
conversations. A plan with findings across multiple categories likely
has a systemic configuration issue, not just isolated data errors.

In [0]:
df_top_plans = (
    df_plan
    .select(
        "plan_id", "plan_name", "tpa_name", "funding_model",
        "affected_members", "categories_affected",
        "distinct_issue_types", "total_dollar_exposure",
        "total_findings",
    )
    .orderBy(F.col("total_dollar_exposure").desc())
    .limit(20)
)

print("Top 20 plans by dollar exposure:")
display(df_top_plans)

log_entry(NB, "plan_exposure", "SUCCESS",
          detail=f"top_plan_exposure=${df_top_plans.first()['total_dollar_exposure']:,.2f}" if df_top_plans.count() > 0 else "no plans")

Top 20 plans by dollar exposure:


plan_id,plan_name,tpa_name,funding_model,affected_members,categories_affected,distinct_issue_types,total_dollar_exposure,total_findings
PLN006,Zeta Manufacturing,TPA 4,SELF_FUNDED,764,4,7,784859.32,1074
PLN026,Beacon Shipping Lines,TPA 8,SELF_FUNDED,523,5,8,569135.74,750
PLN016,Pi Aerospace Defense,TPA 5,SELF_FUNDED,495,4,7,501412.17,672
PLN033,Ironclad Security Services,TPA 2,FULLY_INSURED,418,4,7,438848.09,594
PLN039,Orion Chemical Works,TPA 5,FULLY_INSURED,368,5,9,383202.02,522
PLN032,Harbor Port Authority,TPA 3,SELF_FUNDED,329,4,8,336453.9,466
PLN004,Delta Logistics,TPA 2,SELF_FUNDED,198,4,8,54554.67,199
PLN002,Beta Industries,TPA 1,SELF_FUNDED,173,5,9,43782.4,175
PLN001,Alpha Corp,TPA 1,SELF_FUNDED,176,3,7,40839.11,179
PLN005,Epsilon Teachers Fund,TPA 3,FULLY_INSURED,188,4,8,32186.53,192


[PIPELINE_LOG] {"run_id": "RUN_20261019_160540", "timestamp": "2026-10-19T16:07:04.482582+00:00", "notebook": "NB_08", "section": "plan_exposure", "status": "SUCCESS", "row_count": 0, "detail": "top_plan_exposure=$784,859.32", "duration_sec": 0.0}


## Multi-Category Members

Members who appear in findings from multiple audit categories are the
highest-priority cases for investigation. A member with overlapping
eligibility and accumulator issues, for example, probably has a root
cause in the enrollment feed that's cascading across domains.
Fixing the root cause fixes multiple finding types at once.

In [0]:
# Members with findings in 2+ categories
df_multi = (
    df_member
    .filter(F.col("overlap_count") > 1)
    .select(
        "member_id", "plan_id",
        "overlap_count", "distinct_issue_types",
        "total_findings", "total_dollar_exposure",
        "worst_severity", "categories",
    )
    .orderBy(F.col("overlap_count").desc(), F.col("total_dollar_exposure").desc())
    .limit(25)
)

multi_count = df_member.filter(F.col("overlap_count") > 1).count()
single_count = df_member.filter(F.col("overlap_count") == 1).count()

print(f"Member overlap summary:")
print(f"  Single-category members: {single_count:,}")
print(f"  Multi-category members:  {multi_count:,} (candidates for root cause investigation)")
print()
print("Top 25 multi-category members:")
display(df_multi)

log_entry(NB, "multi_category_members", "SUCCESS",
          detail=f"multi={multi_count}, single={single_count}")

Member overlap summary:
  Single-category members: 5,679
  Multi-category members:  171 (candidates for root cause investigation)

Top 25 multi-category members:


member_id,plan_id,overlap_count,distinct_issue_types,total_findings,total_dollar_exposure,worst_severity,categories
MBR1013278,PLN006,3,3,3,190.81,CRITICAL,"List(ELIGIBILITY, ACCUMULATOR, COPAY_PROGRAM)"
MBR1044176,PLN033,2,2,4,4355.39,CRITICAL,"List(ACCUMULATOR, COPAY_PROGRAM)"
MBR1043687,PLN033,2,2,3,4150.16,CRITICAL,"List(ACCUMULATOR, COPAY_PROGRAM)"
MBR1013079,PLN006,2,2,3,3965.3,CRITICAL,"List(ACCUMULATOR, COPAY_PROGRAM)"
MBR1037089,PLN026,2,2,4,3571.88,CRITICAL,"List(ACCUMULATOR, COPAY_PROGRAM)"
MBR1048337,PLN039,2,2,3,3469.0,CRITICAL,"List(ACCUMULATOR, COPAY_PROGRAM)"
MBR1027639,PLN016,2,2,2,3268.1,CRITICAL,"List(ACCUMULATOR, COPAY_PROGRAM)"
MBR1013632,PLN006,2,2,3,3262.58,CRITICAL,"List(ACCUMULATOR, COPAY_PROGRAM)"
MBR1013776,PLN006,2,2,2,3181.56,CRITICAL,"List(ACCUMULATOR, COPAY_PROGRAM)"
MBR1012546,PLN006,2,2,2,3090.99,CRITICAL,"List(ACCUMULATOR, COPAY_PROGRAM)"


[PIPELINE_LOG] {"run_id": "RUN_20261019_160540", "timestamp": "2026-10-19T16:07:09.466298+00:00", "notebook": "NB_08", "section": "multi_category_members", "status": "SUCCESS", "row_count": 0, "detail": "multi=171, single=5679", "duration_sec": 0.0}


## Overlap Distribution

How many audit categories does a typical affected member span?
If most members have findings in only one category, the issues are
likely independent. If many members span 3+ categories, there is
probably a shared upstream root cause worth investigating.

In [0]:
df_overlap_dist = (
    df_member
    .groupBy("overlap_count")
    .agg(
        F.count("*").alias("members"),
        F.round(F.sum("total_dollar_exposure"), 2).alias("combined_exposure"),
        F.round(F.avg("total_dollar_exposure"), 2).alias("avg_exposure_per_member"),
    )
    .orderBy("overlap_count")
)

print("Distribution of category overlap per member:")
display(df_overlap_dist)

Distribution of category overlap per member:


overlap_count,members,combined_exposure,avg_exposure_per_member
1,5679,3484441.54,613.57
2,170,203581.85,1197.54
3,1,190.81,190.81


## Exposure by Funding Model

Fully-insured vs self-funded plans have different regulatory exposure.
Fully-insured plans in accumulator ban states face state compliance
risk. Self-funded (ERISA) plans are federally preempted but may still
have contractual obligations.

In [0]:
df_plan_design = spark.table("pbm_sim.plan_design")

df_funding = (
    df_registry
    .filter(F.col("plan_id").isNotNull())
    .join(df_plan_design.select("plan_id", "funding_model"), on="plan_id", how="left")
    .groupBy("funding_model")
    .agg(
        F.count("*").alias("findings"),
        F.countDistinct("member_id").alias("members"),
        F.countDistinct("plan_id").alias("plans"),
        F.round(F.sum("dollar_exposure"), 2).alias("dollar_exposure"),
    )
    .orderBy(F.col("dollar_exposure").desc())
)

print("Exposure by plan funding model:")
display(df_funding)

log_entry(NB, "funding_model_analysis", "SUCCESS")

Exposure by plan funding model:


funding_model,findings,members,plans,dollar_exposure
SELF_FUNDED,4823,3947,24,2642226.39
FULLY_INSURED,2259,1905,16,1045987.81


[PIPELINE_LOG] {"run_id": "RUN_20261019_160540", "timestamp": "2026-10-19T16:07:15.861002+00:00", "notebook": "NB_08", "section": "funding_model_analysis", "status": "SUCCESS", "row_count": 0, "detail": "", "duration_sec": 0.0}


## Actionable Next Steps

Based on the findings above, here is a prioritized action list for
the operations and account management teams. The priority order is
driven by severity and dollar exposure.

In [0]:
# Build a prioritized action list from the data.
# This is the "so what" section that turns data into decisions.
print("=" * 65)
print("  RECOMMENDED ACTIONS")
print("=" * 65)

# 1. Critical severity findings
if critical_count > 0:
    crit_exposure = df_registry.filter(F.col("severity") == "CRITICAL").agg(F.sum("dollar_exposure")).collect()[0][0] or 0
    print(f"\n  1. IMMEDIATE: Resolve {critical_count:,} CRITICAL findings (${crit_exposure:,.2f} exposure)")

    # Break down by type
    crit_types = (
        df_registry
        .filter(F.col("severity") == "CRITICAL")
        .groupBy("issue_category", "issue_type")
        .agg(F.count("*").alias("count"), F.round(F.sum("dollar_exposure"), 2).alias("exposure"))
        .orderBy(F.col("exposure").desc())
        .collect()
    )
    for row in crit_types:
        print(f"     - {row['issue_type']}: {row['count']} findings, ${row['exposure']:,.2f}")

# 2. Multi-category members
if multi_count > 0:
    multi_exposure = df_member.filter(F.col("overlap_count") > 1).agg(F.sum("total_dollar_exposure")).collect()[0][0] or 0
    print(f"\n  2. INVESTIGATE: {multi_count:,} members with cross-domain issues (${multi_exposure:,.2f} combined)")
    print(f"     These members have findings in multiple audit categories.")
    print(f"     Root cause analysis on the top cases will likely resolve")
    print(f"     multiple finding types at once.")

# 3. Top plans
top_plan_count = df_plan.filter(F.col("categories_affected") >= 3).count()
if top_plan_count > 0:
    print(f"\n  3. ACCOUNT REVIEW: {top_plan_count} plans have findings across 3+ categories")
    print(f"     Schedule remediation calls with these plan sponsors.")

# 4. Data quality upstream
file_quality_count = df_registry.filter(F.col("issue_category") == "FILE_QUALITY").count()
if file_quality_count > 0:
    print(f"\n  4. UPSTREAM FIX: {file_quality_count:,} file quality findings")
    print(f"     Work with TPA to fix the export process. These issues")
    print(f"     recur every file delivery until the source is fixed.")

print()
print("=" * 65)

log_entry(NB, "action_items", "SUCCESS",
          detail=f"critical={critical_count}, multi_member={multi_count}, "
                 f"high_risk_plans={top_plan_count}, file_quality={file_quality_count}")

  RECOMMENDED ACTIONS

  1. IMMEDIATE: Resolve 2,556 CRITICAL findings ($714,720.89 exposure)
     - DEDUCTIBLE_OVERCHARGE: 1681 findings, $554,713.64
     - OOPM_OVERCHARGE: 375 findings, $160,007.25
     - TERM_MISMATCH: 500 findings, $0.00

  2. INVESTIGATE: 171 members with cross-domain issues ($203,772.66 combined)
     These members have findings in multiple audit categories.
     Root cause analysis on the top cases will likely resolve
     multiple finding types at once.

  3. ACCOUNT REVIEW: 40 plans have findings across 3+ categories
     Schedule remediation calls with these plan sponsors.

  4. UPSTREAM FIX: 32 file quality findings
     Work with TPA to fix the export process. These issues
     recur every file delivery until the source is fixed.

[PIPELINE_LOG] {"run_id": "RUN_20261019_160540", "timestamp": "2026-10-19T16:07:21.912382+00:00", "notebook": "NB_08", "section": "action_items", "status": "SUCCESS", "row_count": 0, "detail": "critical=2556, multi_member=171, hi

## Investigation Notes

This dashboard is intentionally read-only - it queries the findings
registry and rollup tables that NB_07 produces but does not write
any tables itself. If the underlying data changes (a new pipeline run,
a corrected finding), re-running this notebook picks up the latest state.

The action items section is generated from the data, not hard-coded.
As findings get resolved and the registry shrinks, the recommended
actions update automatically.

For scheduled delivery, attach this notebook to the same Databricks
Workflow as the orchestrator and configure email notifications on
completion. The notebook output renders cleanly in the email report.

In [0]:
log_entry(NB, "dashboard_complete", "SUCCESS",
          detail=f"{total_findings} findings, ${total_exposure:,.2f} exposure, "
                 f"{critical_count} CRITICAL, {multi_count} multi-category members")
log_summary(NB)

[PIPELINE_LOG] {"run_id": "RUN_20261019_160540", "timestamp": "2026-10-19T16:07:23.787123+00:00", "notebook": "NB_08", "section": "dashboard_complete", "status": "SUCCESS", "row_count": 0, "detail": "7084 findings, $3,688,214.20 exposure, 2556 CRITICAL, 171 multi-category members", "duration_sec": 0.0}

  PIPELINE LOG SUMMARY: NB_08 (run: RUN_20261019_160540)
  [+] load_dashboard_tables: SUCCESS
  [+] pipeline_health: SUCCESS
  [+] kpi_scorecard: SUCCESS
  [+] severity_distribution: SUCCESS
  [+] category_analysis: SUCCESS
  [+] plan_exposure: SUCCESS
  [+] multi_category_members: SUCCESS
  [+] funding_model_analysis: SUCCESS
  [+] action_items: SUCCESS
  [+] dashboard_complete: SUCCESS

